# 01 — PaySim data profile

This notebook profiles the real PaySim CSV from [`ealaxi/paysim1`](https://www.kaggle.com/datasets/ealaxi/paysim1). It never substitutes the synthetic temporal fixture.

**Questions answered:** Is the snapshot pinned? What is the temporal range? How imbalanced is fraud? Which transaction types and amount ranges dominate? Are required fields missing?

The full CSV is queried with DuckDB so it does not need to be loaded into a pandas DataFrame.

In [1]:
import os
from pathlib import Path

from IPython.display import Markdown, display

from pit_fintech.data.paysim import (
    DATASET_URL,
    connect_paysim,
    find_paysim_csv,
    inspect_snapshot,
    resolve_project_root,
    setup_instructions,
    validate_paysim_csv,
)

PROJECT_ROOT = resolve_project_root(Path.cwd())
paysim_csv = find_paysim_csv(PROJECT_ROOT)
DATA_AVAILABLE = paysim_csv is not None

if DATA_AVAILABLE:
    columns = validate_paysim_csv(paysim_csv)
    print({"dataset": DATASET_URL, "path": str(paysim_csv), "columns": columns})
else:
    display(Markdown("```text\n" + setup_instructions(PROJECT_ROOT) + "\n```"))

{'dataset': 'https://www.kaggle.com/datasets/ealaxi/paysim1', 'path': 'C:\\workspace\\pit-fintech\\data\\raw\\PS_20174392719_1491204439457_log.csv', 'columns': ('step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud')}


In [2]:
# Hashing the full file is intentionally enabled by default because snapshot identity is a gate.
# Set PAYSIM_CHECKSUM=0 only for a quick exploratory rerun.
RUN_CHECKSUM = os.getenv("PAYSIM_CHECKSUM", "1") != "0"

if DATA_AVAILABLE and RUN_CHECKSUM:
    snapshot = inspect_snapshot(paysim_csv)
    print(
        {
            "dataset_snapshot_id": snapshot.dataset_snapshot_id,
            "sha256": snapshot.sha256,
            "size_bytes": snapshot.size_bytes,
            "path": str(snapshot.path),
        }
    )
elif DATA_AVAILABLE:
    print("Checksum skipped for this exploratory run; do not use it as frozen evidence.")
else:
    print("Snapshot inspection skipped until the PaySim CSV is available.")

{'dataset_snapshot_id': 'paysim1:16910f90577b0d98', 'sha256': '16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b', 'size_bytes': 493534783, 'path': 'C:\\workspace\\pit-fintech\\data\\raw\\PS_20174392719_1491204439457_log.csv'}


In [9]:
connection = connect_paysim(paysim_csv) if DATA_AVAILABLE else None


def show(sql: str):
    if connection is None:
        print("Query skipped: PaySim CSV is unavailable.")
        return None
    table = connection.sql(sql).to_arrow_table()
    display(table)

## Core population and temporal coverage

`step` is an hourly ordinal in PaySim, not a real wall-clock timestamp. An absolute timestamp must therefore be a documented synthetic anchor, never presented as source truth.

In [10]:
show(
    """
    SELECT
        count(*) AS rows,
        min(step) AS min_step,
        max(step) AS max_step,
        count(DISTINCT nameOrig) AS origin_entities,
        count(DISTINCT nameDest) AS destination_entities,
        sum(isFraud) AS fraud_rows,
        round(100.0 * avg(isFraud), 6) AS fraud_percent,
        sum(isFlaggedFraud) AS flagged_rows
    FROM paysim
    """
)

pyarrow.Table
rows: int64
min_step: int64
max_step: int64
origin_entities: int64
destination_entities: int64
fraud_rows: decimal128(38, 0)
fraud_percent: double
flagged_rows: decimal128(38, 0)
----
rows: [[6362620]]
min_step: [[1]]
max_step: [[743]]
origin_entities: [[6353307]]
destination_entities: [[2722362]]
fraud_rows: [[8213]]
fraud_percent: [[0.129082]]
flagged_rows: [[16]]

In [11]:
show(
    """
    SELECT
        count_if(step IS NULL) AS step_nulls,
        count_if(type IS NULL) AS type_nulls,
        count_if(amount IS NULL) AS amount_nulls,
        count_if(nameOrig IS NULL OR nameOrig = '') AS origin_nulls,
        count_if(nameDest IS NULL OR nameDest = '') AS destination_nulls,
        count_if(isFraud IS NULL) AS label_nulls,
        count_if(amount < 0) AS negative_amount_rows,
        count_if(isFraud NOT IN (0, 1)) AS invalid_fraud_labels
    FROM paysim
    """
)

pyarrow.Table
step_nulls: decimal128(38, 0)
type_nulls: decimal128(38, 0)
amount_nulls: decimal128(38, 0)
origin_nulls: decimal128(38, 0)
destination_nulls: decimal128(38, 0)
label_nulls: decimal128(38, 0)
negative_amount_rows: decimal128(38, 0)
invalid_fraud_labels: decimal128(38, 0)
----
step_nulls: [[0]]
type_nulls: [[0]]
amount_nulls: [[0]]
origin_nulls: [[0]]
destination_nulls: [[0]]
label_nulls: [[0]]
negative_amount_rows: [[0]]
invalid_fraud_labels: [[0]]

## Transaction and fraud distribution

Accuracy is not useful for this imbalance. Later model evaluation will prioritize PR-AUC and recall at a fixed FPR.

In [12]:
show(
    """
    SELECT
        type,
        count(*) AS rows,
        sum(isFraud) AS fraud_rows,
        round(100.0 * avg(isFraud), 6) AS fraud_percent,
        round(avg(amount), 2) AS mean_amount,
        round(median(amount), 2) AS median_amount
    FROM paysim
    GROUP BY type
    ORDER BY rows DESC, type
    """
)

pyarrow.Table
type: string
rows: int64
fraud_rows: decimal128(38, 0)
fraud_percent: double
mean_amount: double
median_amount: double
----
type: [["CASH_OUT","PAYMENT","CASH_IN","TRANSFER","DEBIT"]]
rows: [[2237500,2151495,1399284,532909,41432]]
fraud_rows: [[4116,0,0,4097,0]]
fraud_percent: [[0.183955,0,0,0.768799,0]]
mean_amount: [[176273.96,13057.6,168920.24,910647.01,5483.67]]
median_amount: [[147072.19,9482.19,143427.71,486308.39,3048.99]]

In [13]:
show(
    """
    SELECT
        isFraud,
        count(*) AS rows,
        round(min(amount), 2) AS min_amount,
        round(quantile_cont(amount, 0.50), 2) AS p50_amount,
        round(quantile_cont(amount, 0.90), 2) AS p90_amount,
        round(quantile_cont(amount, 0.99), 2) AS p99_amount,
        round(max(amount), 2) AS max_amount
    FROM paysim
    GROUP BY isFraud
    ORDER BY isFraud
    """
)

pyarrow.Table
isFraud: int32
rows: int64
min_amount: double
p50_amount: double
p90_amount: double
p99_amount: double
max_amount: double
----
isFraud: [[0,1]]
rows: [[6354407,8213]]
min_amount: [[0.01,0]]
p50_amount: [[74684.72,441423.44]]
p90_amount: [[364373.44,4521723.51]]
p99_amount: [[1586064.17,10000000]]
max_amount: [[92445516.64,10000000]]

In [14]:
show(
    """
    SELECT
        floor((step - 1) / 24)::INTEGER + 1 AS simulation_day,
        count(*) AS rows,
        sum(isFraud) AS fraud_rows,
        round(100.0 * avg(isFraud), 6) AS fraud_percent
    FROM paysim
    GROUP BY simulation_day
    ORDER BY simulation_day
    """
)

pyarrow.Table
simulation_day: int32
rows: int64
fraud_rows: decimal128(38, 0)
fraud_percent: double
----
simulation_day: [[1,2,3,4,5,...,27,28,29,30,31]]
rows: [[574255,455238,1070,28240,9789,...,8578,14661,54890,11287,272]]
fraud_rows: [[271,309,310,262,252,...,280,248,260,268,272]]
fraud_percent: [[0.047192,0.067877,28.971963,0.927762,2.574318,...,3.264164,1.691563,0.473675,2.374413,100]]

## Gate produced by this notebook

- Freeze the full SHA-256 and record `dataset_snapshot_id`.
- Confirm 11 expected columns and binary labels.
- Record the actual row count, hourly range, fraud rate, entity cardinality and per-type fraud coverage.
- Do not lock the entity key or model family here; notebook 02 evaluates temporal/entity viability and notebook 03 audits leakage.